# 06 — Risk-aversion sweep

**Sweep 3.** Fixed canonical instance (n=16, K=4, p=3). Vary `lambda`
across two orders of magnitude. At low `lambda` the problem is
near-linear and greedy wins; at high `lambda` it is covariance-dominated
and frustrated. This sweep tells us *when* QAOA's mechanism could matter.

> **Runtime:** ~15–18 min on Colab CPU from scratch (instant on cache hit).

In [1]:
# === Bootstrap (Colab + local) ===
import os, urllib.request as _u
exec((open('../scripts/bootstrap.py') if os.path.exists('../scripts/bootstrap.py') else _u.urlopen('https://raw.githubusercontent.com/egil10/fys5419/main/project2/code/scripts/bootstrap.py')).read())

# === Project imports ===
import json
import numpy as np
import pandas as pd

from scripts.colab     import out_dir
from scripts.data      import load_universe
from scripts.portfolio import PortfolioProblem, DEFAULTS
from scripts.classical import brute_force, greedy_top_k, markowitz_round, simulated_annealing
from scripts.qaoa      import solve, make_hamiltonians
from scripts.metrics   import prob_optimal, scaled_ratio, gap

RESULTS = out_dir('results')
print(f'Results will be saved to: {RESULTS}')

> /usr/bin/python3 -m pip install -q numpy pandas scipy matplotlib yfinance pyarrow
[colab.setup] env=Colab
[colab.setup] cwd  = /content/fys5419/project2/code/notebooks
[colab.setup] path = /content/fys5419/project2/code  (added to sys.path)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results will be saved to: /content/drive/MyDrive/GITHUB-COLAB/fys5419/project2/code/results


In [ ]:
# === Canonical instance: n=16, K=4 ===
K       = DEFAULTS['K_AT_16']
AP      = DEFAULTS['A']
P       = 3
N_SEEDS = 10

LAMBDAS = np.logspace(-1, 2, 7)  # 0.1, ..., 100

In [3]:
r = load_universe()
cache = RESULTS / 'risk_sweep.json'

if cache.exists():
    rows = json.loads(cache.read_text())
    # Same legacy-format guard as the size-scaling sweep.
    _ratios = [r_.get('ratio', 1.0) for r_ in rows]
    if _ratios and (min(_ratios) < -0.5 or max(_ratios) > 1.5):
        raise RuntimeError(
            f'{cache.name} contains ratios outside [-0.5, 1.5] '
            f'(min={min(_ratios):.2f}, max={max(_ratios):.2f}) — this is the '
            f'legacy cost/E_opt ratio format. Delete the file and rerun.'
        )
    print(f'loaded {cache.name}')
else:
    rows = []
    for lam in LAMBDAS:
        pf = PortfolioProblem(r.mu, r.Sigma, lam=float(lam), A=AP, K=K,
                              tickers=r.tickers)
        bf = brute_force(pf)

        # Spectrum endpoints in COST units. make_hamiltonians returns HC + offset
        # so HC.min() == bf.cost; do NOT use from_portfolio, that drops the AK^2
        # offset and would shift the ratio.
        _, _, _, _, _, HC = make_hamiltonians(pf)
        e_opt   = float(HC.min())
        e_worst = float(HC.max())

        gr = greedy_top_k(pf)
        mr = markowitz_round(pf)
        sa_runs = [simulated_annealing(pf, n_sweeps=1000, seed=s) for s in range(N_SEEDS)]
        sa_med  = float(np.median([sr.cost for sr in sa_runs]))

        qres = solve(pf, p=P, n_restarts=N_SEEDS, seed=42)

        def row(name, cost, extra=None):
            r_ = {'lambda':  float(lam),
                  'solver':  name,
                  'cost':    float(cost),
                  'ratio':   scaled_ratio(cost, e_opt, e_worst),  # in [0,1]
                  'gap_rel': gap(cost, e_opt),                    # 0 = optimal
                  'e_opt':   e_opt,
                  'e_worst': e_worst}
            if extra: r_.update(extra)
            return r_

        rows.extend([
            row('brute_force',     bf.cost),
            row('greedy_sharpe',   gr.cost),
            row('markowitz_round', mr.cost),
            row('sa_median',       sa_med),
            row(f'qaoa_p{P}',      float(qres['energy']),
                extra={'p_optimal': prob_optimal(qres['probs'], bf.x)}),
        ])
        print(f'  lambda={lam:6.2f}: brute={bf.cost:+.4f}  '
              f'qaoa_ratio={scaled_ratio(qres["energy"], e_opt, e_worst):.4f}  '
              f'p_opt={prob_optimal(qres["probs"], bf.x):.3f}')

    cache.write_text(json.dumps(rows, indent=2))
    print(f'saved -> {cache.name}')

pd.DataFrame(rows)

  lambda=  0.10: brute=-0.0107  qaoa_ratio=0.9501  p_opt=0.000
  lambda=  0.32: brute=-0.0071  qaoa_ratio=0.9491  p_opt=0.000
  lambda=  1.00: brute=-0.0045  qaoa_ratio=0.9488  p_opt=0.000
  lambda=  3.16: brute=+0.0007  qaoa_ratio=0.9300  p_opt=0.000
  lambda= 10.00: brute=+0.0103  qaoa_ratio=0.9433  p_opt=0.000
  lambda= 31.62: brute=+0.0390  qaoa_ratio=0.9697  p_opt=0.000
  lambda=100.00: brute=+0.1286  qaoa_ratio=0.9541  p_opt=0.000
saved -> risk_sweep.json


,lambda,solver,cost,ratio,gap_rel,e_opt,e_worst,p_optimal
0,0.100000,brute_force,-0.010653,1.000000,-1.221245e-14,-0.010653,71.983823,NaN
1,0.100000,greedy_sharpe,-0.007232,0.999952,3.211269e-01,-0.010653,71.983823,NaN
2,0.100000,markowitz_round,-0.007232,0.999952,3.211269e-01,-0.010653,71.983823,NaN
3,0.100000,sa_median,-0.010292,0.999995,3.396285e-02,-0.010653,71.983823,NaN
4,0.100000,qaoa_p3,3.583778,0.950074,3.373972e+02,-0.010653,71.983823,0.000133
5,0.316228,brute_force,-0.007097,1.000000,1.500745e-13,-0.007097,72.009103,NaN
6,0.316228,greedy_sharpe,-0.006569,0.999993,7.442247e-02,-0.007097,72.009103,NaN
7,0.316228,markowitz_round,-0.006569,0.999993,7.442247e-02,-0.007097,72.009103,NaN
8,0.316228,sa_median,-0.006968,0.999998,1.818932e-02,-0.007097,72.009103,NaN
9,0.316228,qaoa_p3,3.660890,0.949067,5.168161e+02,-0.007097,72.009103,0.000122
